In [2]:
from dotenv import load_dotenv
import openai
import requests
import os
load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


PROMPT = """
너는 유능한 영화 전문 AI를 만들거다.
여기서 사용할 모델은 "gpt-4o-mini" 사용하도록 한다.
영화 관련 소스는 모두 BASE_URL에서 가져오도록 한다.


이제 세가지의 함수를 만들어야한다.

get_popular_movies() - /movies에서 인기 영화를 가져옵니다.
get_movie_details(id) - /movies/:id에서 영화 정보를 가져옵니다.
get_similar_movies(id) - /movies/:id/similar에서 유사한 영화를 조회합니다.


이 세가지 함수를 꼭 사용해서 정보를 가져오도록 한다.
모델이 함수명과 인자(Arguments)를 올바르게 출력해야한다.


"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": PROMPT}],
)

message = response.choices[0].message.content
print(message)

아래는 요청하신 세 가지 함수(`get_popular_movies()`, `get_movie_details(id)`, `get_similar_movies(id)`)를 구현한 예시입니다. 이 함수들은 각각 인기 영화, 특정 영화의 상세 정보, 그리고 유사한 영화를 가져오는 역할을 합니다. 기본적으로 API 호출을 통해 정보를 가져오는 형태로 작성되었습니다.

```python
import requests

BASE_URL = "http://example.com/api"  # BASE_URL을 알맞은 API URL로 설정하세요.

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    if response.status_code == 200:
        return response.json()  # 인기 영화 목록을 반환
    else:
        return {"error": "Unable to fetch popular movies."}

def get_movie_details(movie_id):
    response = requests.get(f"{BASE_URL}/movies/{movie_id}")
    if response.status_code == 200:
        return response.json()  # 특정 영화의 세부정보를 반환
    else:
        return {"error": f"Unable to fetch movie details for ID: {movie_id}"}

def get_similar_movies(movie_id):
    response = requests.get(f"{BASE_URL}/movies/{movie_id}/similar")
    if response.status_code == 200:
        return response.json()  # 유사한 영화 목록을 반환
    else:
      

In [4]:
from openai.types.chat import ChatCompletionMessage
import requests
import os
import openai
import json
from dotenv import load_dotenv

# 환경 변수를 로드합니다.
load_dotenv()
client = openai.OpenAI()
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
messages=[]
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    if response.status_code == 200:
        return response.json()  # 인기 영화 목록을 반환
    else:
        return {"error": "Unable to fetch popular movies."}

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    if response.status_code == 200:
        return response.json()  # 특정 영화의 세부정보를 반환
    else:
        return {"error": f"Unable to fetch movie details for ID: {id}"}

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    if response.status_code == 200:
        return response.json()  # 유사한 영화 목록을 반환
    else:
        return {"error": f"Unable to fetch similar movies for ID: {id}"}

In [5]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_popular_movies",
        "description": "인기 영화 목록을 가져옵니다.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }},
    {"type": "function", "function": {
        "name": "get_movie_details",
        "description": "특정 ID의 영화 상세 정보를 가져옵니다.",
        "parameters": {"type": "object", "properties": {
            "id": {"type": "integer", "description": "영화 ID"}
        }, "required": ["id"]},
    }},
    {"type": "function", "function": {
        "name": "get_similar_movies",
        "description": "특정 ID의 영화와 유사한 영화를 조회합니다.",
        "parameters": {"type": "object", "properties": {
            "id": {"type": "integer", "description": "영화 ID"}
        }, "required": ["id"]},
    }},
]


FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_similar_movies': get_similar_movies,
}

In [6]:
def _tool_result_to_str(result) -> str:
    if isinstance(result, str):
        return result
    return json.dumps(result, ensure_ascii=False)


def process_ai_response(message: ChatCompletionMessage) -> bool:
    """대화 기록에 assistant / tool 메시지를 추가합니다. 도구 호출이 있으면 True."""
    if message.tool_calls:
        assistant_msg = {
            "role": "assistant",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
                for tool_call in message.tool_calls
            ],
        }
        if message.content:
            assistant_msg["content"] = message.content
        messages.append(assistant_msg)

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            raw_args = tool_call.function.arguments

            print(f"Calling function: {function_name} with {raw_args}")

            try:
                arguments = json.loads(raw_args)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            if function_to_run is None:
                result = {"error": f"Unknown function: {function_name}"}
            else:
                result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": _tool_result_to_str(result),
                }
            )
        return True

    messages.append({"role": "assistant", "content": message.content})
    print(f"AI: {message.content}")
    return False

In [7]:
def call_ai():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message
        had_tools = process_ai_response(msg)
        if not had_tools:
            break


while True:
    message = input("Send a message to the LLM..")
    if message == "quit" or message == "q":
        break
    messages.append({"role": "user", "content": message})
    print(f"User: {message}")
    call_ai()

User: 가장 인기있는 영화가 뭐야?
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a resulf of [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/1x9e0qWonw634NhIsRdvnneeqvN.jpg', 'genre_ids': [10749, 18], 'id': 1523145, 'original_language': 'ru', 'original_title': 'Твоё сердце будет разбито', 'overview': 'High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but her family and classmates have reasons to separate the lovers.', 'popularity': 1002.4008, 'poster_path': 'https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg', 'release_date': '2026-03-26', 'title': 'Your Heart Will Be Broken', 'video': False, 'vote_average': 6.75, 'vote_count': 54}, {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/iN41Ccw4DctL8npfmYg1j5Tr1eb.

BadRequestError: Error code: 400 - {'error': {'message': "Missing required parameter: 'messages[2].content[0].type'.", 'type': 'invalid_request_error', 'param': 'messages[2].content[0].type', 'code': 'missing_required_parameter'}}